# Simulation: Equal ETF Buy & Hold

This notebook simulates the **Equal ETF Buy & Hold Strategy**.

- **Portfolio:** 33.3% SPY, 33.3% QQQ, 33.3% VTI
- **Strategy:** Buy on day 1 and never sell. Add $500 every 10 trading days and buy more shares.
- **Initial Investment:** $10,000
- **Biweekly Contribution:** Add $500 every 10 trading days and buy more shares.

### How Buy & Hold Works
- Day 1: Invest $10,000 → buy SPY shares
- Day 10: Add $500 and buy more shares: 33.3% SPY, 33.3% QQQ, 33.3% VTI
- Day 20: Add $500 and buy more SPY shares: 33.3% SPY, 33.3% QQQ, 33.3% VTI
- Day 30: Add $500 and buy more SPY shares: 33.3% SPY, 33.3% QQQ, 33.3% VTI
... 

---

## 1. Import Libraries, Configure Settings & Initialize Variables

In [ ]:
import pandas as pd
import os

PROCESSED_DIR   = "../data/processed"
SIMULATIONS_DIR = "../data/simulations"
os.makedirs(SIMULATIONS_DIR, exist_ok=True)
INPUT_FILE  = os.path.join(PROCESSED_DIR, "prices_with_indicators.csv")
OUTPUT_FILE = os.path.join(SIMULATIONS_DIR, "equal_buy_hold.csv")

initial_investment = 10000 # Starting investment on day 1
contribution_amount = 500 # Amount added every 10 trading days
contribution_interval = 10 # Every 10 trading days (biweekly)

# weight =  percentage of money going into each ETF expressed as a decimal
weight_spy = 1/3
weight_qqq = 1/3
weight_vti = 1/3

# Strategy/portfolio labels 
STRATEGY = "Buy & Hold"
PORTFOLIO_TYPE = "Equal"

print(f"Input: {os.path.abspath(INPUT_FILE)}")
print(f"Output: {os.path.abspath(OUTPUT_FILE)}")
print(f"\nSimulation Parameters:")
print(f"Initial Investment: ${initial_investment:,}")
print(f"Contribution Amount: ${contribution_amount}")
print(f"Contribution Interval: Contributions are made every {contribution_interval} trading days")
print(f"Portfolio: SPY={weight_spy*100}%, QQQ={weight_qqq*100}%, VTI={weight_vti*100}%")
print(f"Strategy: {STRATEGY}")
print(f"Portfolio Type: {PORTFOLIO_TYPE}")

## 2. Load `prices_with_indicators.csv`
Note:
- The first 49 rows have `NaN` in the MA50 columns — this is expected and correct.  
- Buy & Hold ignores MA50 entirely so these NaN values do not affect the simulation.

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["Date"])

# Confirm the file loaded correctly
print(f"Loaded prices_with_indicators.csv")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Nulls: {df.isnull().sum().to_dict()}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string(index=False))

## 3. Simulate Equal ETF Buy & Hold Strategy

The simulation loops through each row through every trading day in the dataset.  
The logic is identical to the Single simulation, but cash is now split three ways on each buy.

**Row 0: The Initial Investment:**
- cash = $10,000
- total_invested = $10,000
- cash = 0

**Every row where i > 0 and i % 10 == 0 (Contribution Days):**
- contribution = $500
- cash += $500
- total_invested += $500
- shares_spy += (cash * 0.333) / SPY_Price
- shares_qqq += (cash * 0.333) / QQQ_Price
- shares_vti += (cash * 0.333) / VTI_Price
- cash = 0

### For Buy & Hold:
- All three `Position_` columns are always **1**
- `Cash` is always **0** after shares are bought
- `SPY_MA50`, `QQQ_MA50`, `VTI_MA50` are left blank (NaN)

In [ ]:
shares_spy = 0.0
shares_qqq = 0.0
shares_vti = 0.0
cash = 0.0
total_invested = 0.0

results = []

for i, row in df.iterrows():
    spy_price = row["SPY_Price"]
    qqq_price = row["QQQ_Price"]
    vti_price = row["VTI_Price"]

    contribution = 0.0  
    
    # Initial Investment 
    if i == 0:
        cash = initial_investment
        total_invested = initial_investment

    elif i % contribution_interval == 0:
        contribution = contribution_amount
        cash += contribution
        total_invested += contribution


    if cash > 0:
        
        # SPY: 33.3% of available cash
        amount_spy  = cash * weight_spy
        shares_spy += amount_spy / spy_price
        
     # QQQ: 33.3% of available cash
        amount_qqq  = cash * weight_qqq
        shares_qqq += amount_qqq / qqq_price

        # VTI: 33.3% of available cash
        amount_vti  = cash * weight_vti
        shares_vti += amount_vti / vti_price

        # Cash goes to 0 after all shares are bought
        cash = 0.0
        
    portfolio_value = (
        (shares_spy * spy_price) +
        (shares_qqq * qqq_price) +
        (shares_vti * vti_price) +
        cash 
    )

    results.append({
        "Date": row["Date"],
        "SPY_Price": spy_price,
        "QQQ_Price": qqq_price,
        "VTI_Price": vti_price,
        "SPY_MA50": None,
        "QQQ_MA50": None,      
        "VTI_MA50": None,     
        "Position_SPY": 1, 
        "Position_QQQ": 1,
        "Position_VTI": 1, 
        "Contribution": contribution,
        "Cash": cash,  
        "Shares_SPY": shares_spy,
        "Shares_QQQ": shares_qqq, 
        "Shares_VTI": shares_vti,
        "Total_Invested": total_invested,
        "Portfolio_Value":portfolio_value,
        "Earnings": None,  
        "Daily_Return": None, 
        "Strategy": STRATEGY,
        "Portfolio_Type": PORTFOLIO_TYPE
    })

result_df = pd.DataFrame(results)
print(f"Simulation complete. Rows processed: {len(result_df)}")

## 4. Calculate Earnings and Daily Return

- Earnings = Portfolio_Value - Total_Invested

- Daily_Return ( percentage change in portfolio value from the previous day)
- = Portfolio_Value.pct_change()
= (today - yesterday) / yesterday

- The first row of `Daily_Return` will always be `NaN` because there is no previous day to compare to.

In [ ]:
result_df["Earnings"] = result_df["Portfolio_Value"] - result_df["Total_Invested"]
result_df["Daily_Return"] = result_df["Portfolio_Value"].pct_change()

print("Earnings and Daily Return calculated.")
print(result_df[["Date", "Portfolio_Value", "Total_Invested", "Earnings", "Daily_Return"]]
      .head(15).to_string(index=False))

## 5. Save `equal_buy_hold.csv`

Save the simulation results to `data/simulations/`.  
The file will have exactly 21 columns as defined in the data dictionary.

In [ ]:
final_cols = [
    "Date", "SPY_Price", "QQQ_Price", "VTI_Price",
    "SPY_MA50", "QQQ_MA50", "VTI_MA50",
    "Position_SPY", "Position_QQQ", "Position_VTI",
    "Contribution", "Cash",
    "Shares_SPY", "Shares_QQQ", "Shares_VTI",
    "Total_Invested", "Portfolio_Value", "Earnings",
    "Daily_Return", "Strategy", "Portfolio_Type"
]

result_df = result_df[final_cols]
result_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved equal_buy_hold.csv")
print(f"Path: {os.path.abspath(OUTPUT_FILE)}")
print(f"Rows: {len(result_df)}")
print(f"Columns: {len(result_df.columns)} —> {list(result_df.columns)}")
print(f"\nFinal Portfolio Summary")
print(f"Total Invested: ${result_df['Total_Invested'].iloc[-1]:,.2f}")
print(f"Final Value: ${result_df['Portfolio_Value'].iloc[-1]:,.2f}")
print(f"Total Earnings: ${result_df['Earnings'].iloc[-1]:,.2f}")